# Stage 1: Data Understanding

This notebook validates the runtime data acquisition layer for Phase 1. It downloads data directly from Yahoo Finance, drops assets with more than 5% missing observations, forward-fills the remaining gaps, back-fills any leading gaps, inspects adjusted close prices and volume, and visualizes the sample asset universe.

## Objectives

- Use the `YahooFinanceProvider` directly at runtime.
- Download the sample universe: `HDFCBANK.NS`, `TCS.NS`, `GOLDBEES.NS`.
- Validate dates, adjusted close coverage, volume coverage, and missing values.
- Drop assets with more than 5% missing observations, then forward-fill and back-fill the remaining gaps.
- Produce in-memory outputs: `prices_df` and `volume_df`.
- Visualize price behavior and data completeness before moving to returns.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

project_root = Path.cwd().resolve()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import src.data_pipeline.preprocess as preprocess_module
import src.data_pipeline as data_pipeline_module
importlib.reload(preprocess_module)
importlib.reload(data_pipeline_module)

from src.data_pipeline import SAMPLE_UNIVERSE, DataPreprocessor, MarketDataBundle, YahooFinanceProvider, build_data_inspection_table

In [ ]:
symbols = list(SAMPLE_UNIVERSE)
start_date = "2019-01-01"
end_date = "2020-12-31"

provider = YahooFinanceProvider()
market_data = provider.get_market_data(symbols, start_date, end_date)

raw_prices_df = market_data.prices_df.copy()
raw_volume_df = market_data.volume_df.copy()

prices_df, price_quality_report = DataPreprocessor.handle_missing_values(raw_prices_df)
volume_df = raw_volume_df.reindex(columns=prices_df.columns)
dropped_assets = list(price_quality_report.dropped_asset_names)
cleaned_market_data = MarketDataBundle(
    prices_df=prices_df,
    volume_df=volume_df,
    raw_data=market_data.raw_data,
    price_field=market_data.price_field,
)
inspection_df = build_data_inspection_table(cleaned_market_data)

print(f"Price field used: {market_data.price_field}")
print(f"Raw price shape: {raw_prices_df.shape}")
print(f"Raw volume shape: {raw_volume_df.shape}")
print(f"Clean price shape: {prices_df.shape}")
print(f"Clean volume shape: {volume_df.shape}")
print(f"Dropped price assets (>5% missing): {price_quality_report.assets_dropped}")
if dropped_assets:
    print(", ".join(dropped_assets))
print(f"Price missing before cleaning: {price_quality_report.missing_before}")
print(f"Price missing after cleaning: {price_quality_report.missing_after}")

In [ ]:
prices_df.head()

In [ ]:
volume_df.head()

In [ ]:
inspection_df

In [ ]:
prices_df.describe().T

In [ ]:
volume_df.describe().T

In [ ]:
missing_summary = pd.DataFrame(
    {
        "price_missing_before": raw_prices_df.isna().sum(),
        "price_missing_after": prices_df.isna().sum(),
        "price_missing_before_ratio": raw_prices_df.isna().mean(),
        "price_missing_after_ratio": prices_df.isna().mean(),
    }
)
missing_summary.sort_values("price_missing_before_ratio", ascending=False)

In [ ]:
sns.set_theme(style="whitegrid")
normalized_prices = prices_df / prices_df.iloc[0]
ax = normalized_prices.plot(figsize=(11, 5), linewidth=2)
ax.set_title("Normalized Adjusted Close Prices")
ax.set_ylabel("Growth of 1.0")
ax.set_xlabel("Date")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
raw_prices_df.isna().astype(int).T.pipe(sns.heatmap, cmap="Reds", cbar=False, ax=axes[0])
axes[0].set_title("Missing Values in raw_prices_df (Before Cleaning)")
axes[0].set_ylabel("Asset")

prices_df.isna().astype(int).T.pipe(sns.heatmap, cmap="Greens", cbar=False, ax=axes[1])
axes[1].set_title("Missing Values in prices_df (After Cleaning)")
axes[1].set_ylabel("Asset")
axes[1].set_xlabel("Observation Index")
plt.tight_layout()

## Interpretation

- `prices_df` is the adjusted close panel used for downstream return calculations.
- `volume_df` helps validate tradability and data completeness before portfolio construction.
- Any missing-value clusters should be handled before returns are computed in Stage 2.
- This stage deliberately keeps all processing in memory and avoids persistent dataset storage.